# 01 Exploration of the gov.br reviews

First look at the real Google Play data (build step 2). This is **not** the drift reference
snapshot nor the annotation set: those are decided after this notebook.

**Privacy.** This notebook reads `data/interim/` only, where author name, profile picture and reply
text are already removed. Outputs can still show review text, so they are stripped before every
commit (nbstripout); never paste outputs elsewhere with authorship.

**How to read it.** Each analysis has a short *Concept* cell first: the statistical or data idea
behind it and why it matters for this project. Numbers worth carrying forward are collected in the
last section.

In [ ]:
import json
import re
import unicodedata
from collections import Counter
from itertools import pairwise

import numpy as np
import pandas as pd
import plotly.express as px

from govbr_classifier.models.keyword_baseline import normalize, predict_tema
from govbr_classifier.settings import PROJECT_ROOT

INTERIM = PROJECT_ROOT / "data" / "interim"
df = pd.read_parquet(INTERIM / "reviews.parquet")
meta = json.loads((INTERIM / "reviews.metadata.json").read_text())
SCORES = [1, 2, 3, 4, 5]

# Review text can itself carry personal data (names, CPF numbers). Printed examples skip any text
# matching these patterns; section 5b counts them without showing them.
PERSONAL_DATA = {
    "cpf or cpf-like number": re.compile(r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b|\bcpf\b", re.I),
    "phone-like number": re.compile(r"\(?\b\d{2}\)?\s?9?\d{4}-?\d{4}\b"),
    "email": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "self-introduction with name": re.compile(r"\b(?:me chamo|meu nome [eé])\b", re.I),
}


def has_personal_data(value):
    return any(pattern.search(value) for pattern in PERSONAL_DATA.values())


def printable(series):
    # Examples safe to print: texts without personal-data patterns.
    return series[~series.map(has_personal_data)]


pd.Series(
    {
        "complete_history": meta["complete_history"],
        "rows_raw": meta["stats"]["rows_raw"],
        "rows_interim": meta["stats"]["rows_interim"],
        "duplicate_rate_by_id": f"{meta['stats']['duplicate_rate']:.4%}",
        "ids_edited_during_collection": meta["stats"]["ids_edited_during_collection"],
        "oldest_review_utc": meta["stats"]["review_window_utc"]["oldest"],
        "newest_review_utc": meta["stats"]["review_window_utc"]["newest"],
    }
).to_frame("value")

## 1. Volume per month

**Concept: a count over time is a time series, and its shape is mostly about who shows up.**
Review volume is not a measure of how many people use the app; it measures how many chose to write.
Spikes usually come from events: a new version, a public service that suddenly requires gov.br, a
news story. Think of a restaurant's comment book: a busy holiday fills pages without the food
changing. For this project, volume matters twice: it tells us which periods dominate the training
data (a model learns the loudest months best), and it is the baseline the drift monitor will
compare against.

Two caveats specific to this source. The review date is the **last edit**, so old reviews that
were edited later move to the right. And the app was listed on Google Play on 2020-04-27, so
March 2020 (start of the pandemic) is necessarily empty: the marker is there for context.

In [ ]:
monthly = df.set_index("at").resample("MS").size().rename("reviews").reset_index()
fig = px.bar(monthly, x="at", y="reviews", title="Reviews per month (date of last edit, UTC)")
events = [
    ("2020-03-01", "Pandemic starts (Mar 2020)"),
    ("2020-04-27", "Listed on Google Play"),
    ("2021-09-17", "App redesign"),
]
for i, (day, label) in enumerate(events):
    fig.add_vline(x=pd.Timestamp(day).timestamp() * 1000, line_dash="dash", line_color="gray")
    fig.add_annotation(
        x=day,
        y=1 - 0.08 * i,
        yref="paper",
        text=label,
        showarrow=False,
        xanchor="left",
        bgcolor="rgba(255,255,255,0.7)",
    )
fig.update_layout(xaxis_title=None, yaxis_title="reviews")
fig.show()
monthly.sort_values("reviews", ascending=False).head(5)

## 2. Rating distribution

**Concept: review ratings are J-shaped because of self-selection.** People who loved or hated
something are the ones who write, so 5 and 1 stars dominate and 2 to 4 are thin. The distribution
describes the writers, not the users. Analogy: a complaints box plus a thank-you card box, with no
box for "it was fine". Why it matters here: the rating is logged for monitoring and **never used as
a feature** (ADR-01). If it were, the model would learn "1 star means problem", which is a shortcut
that breaks the moment someone gives 5 stars and still describes a bug (section 8).

In [ ]:
counts = df["score"].value_counts().reindex(SCORES).rename("reviews").to_frame()
counts["share"] = counts["reviews"] / counts["reviews"].sum()
fig = px.bar(
    counts.reset_index(),
    x="score",
    y="share",
    text_auto=".1%",
    title="Share of reviews by star rating",
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

by_year = (
    df.assign(year=df["at"].dt.year)
    .groupby(["year", "score"])
    .size()
    .groupby(level=0)
    .transform(lambda s: s / s.sum())
    .rename("share")
    .reset_index()
)
fig = px.bar(
    by_year,
    x="year",
    y="share",
    color="score",
    title="Rating mix by year",
    category_orders={"score": SCORES},
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()
counts

## 3. Text length in characters and words, by rating

**Concept: percentiles describe a skewed distribution better than the mean.** Text lengths have a
long right tail: most reviews are short, a few are long, and the mean is pulled toward the tail. The
median (p50) is the typical review; p90 and p99 tell where "long" begins. Analogy: a shoe store
stocks by size percentiles, not by the average foot. Why it matters: **the p99 in characters goes
into the API contract** (ADR-03). Text above it gets the warning `length_above_training_p99`, the
honest way to say "longer than almost anything the model saw". Google Play caps reviews at 500
characters, which shows up as a pile at the right edge of the histogram.

In [ ]:
text = df["text"].fillna("")
df["chars"] = text.str.len()
df["words"] = text.str.split().str.len()
QUANTILES = [0.10, 0.50, 0.90, 0.99]


def percentile_table(column):
    table = df.groupby("score")[column].quantile(QUANTILES).unstack()
    table.columns = [f"p{int(q * 100)}" for q in QUANTILES]
    table.loc["all"] = df[column].quantile(QUANTILES).to_numpy()
    table["mean"] = [*df.groupby("score")[column].mean().tolist(), df[column].mean()]
    table["max"] = [*df.groupby("score")[column].max().tolist(), df[column].max()]
    return table.round(1)


P99_CHARS = float(df["chars"].quantile(0.99))
print(f"p99 length in characters (all reviews): {P99_CHARS:.0f}")
print(f"share of reviews at the 500-character cap: {(df['chars'] >= 500).mean():.2%}")
print("--------------------------------------------------")
print("percentile_table('chars'):")
display(percentile_table("chars"))
print("--------------------------------------------------")
print("percentile_table('words'):")
display(percentile_table("words"))

**Concept: in a right-skewed distribution, mode < median < mean.** The mode is the most common
value, the median splits the reviews in half, and the mean is the balance point. A long right tail
does not move the mode, barely moves the median, and pulls the mean toward itself, because every
long review adds its full length to the sum. Analogy: one billionaire walking into a bar moves the
average income of the room, not the typical customer. Here: the mode is the first bar (0 to 9
characters, about 362 thousand reviews, 45% of the data), the median is 12 characters, the mean
is 43. The mean describes almost no real review, which is why the contract uses percentiles. The
tail is also why the histogram needs a log scale: on a linear axis the bars past 200 characters
are a few pixels tall, and the pile just below the 500-character cap (about 2,100 reviews between
490 and 499) disappears next to the first bar.

In [ ]:
# The last bin, [500, 510), holds the reviews at Google Play's 500-character cap.
AXIS_MAX = 510
bins = np.arange(0, AXIS_MAX + 10, 10)
hist = (
    pd.cut(df["chars"], bins=bins, right=False)
    .value_counts(sort=False)
    .rename("reviews")
    .reset_index()
)
hist["chars"] = [interval.left for interval in hist["chars"]]
beyond = int((df["chars"] >= AXIS_MAX).sum())
beyond_note = (
    f"{beyond:,} reviews ({beyond / len(df):.3%}) are longer than {AXIS_MAX} and not shown"
)
print(beyond_note)
print(
    f"mode bin: {int(hist.loc[hist['reviews'].idxmax(), 'chars'])} to "
    f"{int(hist.loc[hist['reviews'].idxmax(), 'chars']) + 9} chars, "
    f"median: {df['chars'].median():.0f}, mean: {df['chars'].mean():.1f}"
)

markers = {
    f"median = {df['chars'].median():.0f}": (df["chars"].median(), "seagreen"),
    f"mean = {df['chars'].mean():.0f}": (df["chars"].mean(), "darkorange"),
    f"p99 = {P99_CHARS:.0f}": (P99_CHARS, "firebrick"),
}
for scale in ["linear", "log"]:
    fig = px.bar(
        hist,
        x="chars",
        y="reviews",
        log_y=scale == "log",
        title=f"Length in characters, bins of 10, {scale} axis<br><sup>{beyond_note}</sup>",
    )
    for row, (label, (x, color)) in enumerate(markers.items()):
        fig.add_vline(x=x, line_dash="dash", line_color=color)
        fig.add_annotation(
            x=x,
            y=1 - 0.07 * row,
            yref="paper",
            text=label,
            showarrow=False,
            xanchor="left",
            font_color=color,
        )
    fig.show()

In [ ]:
long = df.melt(id_vars="score", value_vars=["chars", "words"], var_name="unit")
stats = long.groupby(["unit", "score"])["value"].quantile([0.5, 0.9, 0.99]).unstack().reset_index()
stats.columns = ["unit", "score", "p50", "p90", "p99"]
# The words panel must show word counts, not characters: check it against the table above.
assert (
    stats.set_index(["unit", "score"]).loc[("words", 1), "p99"]
    == percentile_table("words").loc[1, "p99"]
)
fig = px.bar(
    stats.melt(id_vars=["unit", "score"], var_name="percentile"),
    x="score",
    y="value",
    color="percentile",
    barmode="group",
    facet_col="unit",
    text_auto=".0f",
    title="Length percentiles by rating (each panel has its own vertical axis)",
)
# Facets share one axis by default and hide the tick labels of the right panel. Unshared
# without labels, the words bars fill the panel and read against the characters axis.
fig.update_yaxes(matches=None, showticklabels=True)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(title_text="characters", col=1)
fig.update_yaxes(title_text="words", col=2)
fig.update_xaxes(tickmode="linear", dtick=1)
fig.show()

## 4. App versions

**Concept: a high-cardinality categorical variable, and a natural drift covariate.** There are many
versions, each with few reviews, and the mix changes over time as old versions disappear. Version is
useful context for monitoring (a new release often brings a new kind of complaint) but a poor model
feature: a version the model never saw tells it nothing. Remember the limitation from
`data/README.md`: the field may be the version at the last edit, and it is sometimes empty.

In [ ]:
versions = df["app_version"].replace("", np.nan)
print(f"reviews without app version: {versions.isna().mean():.1%}")
print(f"distinct versions: {versions.nunique()}")
top = versions.value_counts().head(20).rename("reviews").reset_index()
fig = px.bar(top, x="reviews", y="app_version", orientation="h", title="20 most reviewed versions")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

# The collection ends mid-quarter, so the last quarter holds fewer days than the others and its
# count is not comparable. It gets its own label and a shaded band in every quarterly chart.
LAST_REVIEW = df["at"].max()
quarter = df["at"].dt.tz_localize(None).dt.to_period("Q")
LAST_QUARTER = quarter.max()
covered_days = (LAST_REVIEW.tz_localize(None).normalize() - LAST_QUARTER.start_time).days + 1
total_days = (LAST_QUARTER.end_time - LAST_QUARTER.start_time).days + 1
PARTIAL_LABEL = f"{LAST_QUARTER} (partial)"
PARTIAL_NOTE = (
    f"{LAST_QUARTER} is partial: {covered_days} of {total_days} days, "
    f"collection ends {LAST_REVIEW:%Y-%m-%d}"
)
df["quarter"] = quarter.astype(str).replace(str(LAST_QUARTER), PARTIAL_LABEL)
QUARTERS = sorted(df["quarter"].unique())


def mark_partial_quarter(fig):
    fig.update_xaxes(type="category", categoryorder="array", categoryarray=QUARTERS)
    position = QUARTERS.index(PARTIAL_LABEL)
    fig.add_vrect(x0=position - 0.5, x1=position + 0.5, fillcolor="grey", opacity=0.2, line_width=0)
    fig.add_annotation(
        x=1,
        y=1.02,
        xref="paper",
        yref="paper",
        text=PARTIAL_NOTE,
        showarrow=False,
        xanchor="right",
        yanchor="bottom",
        font_size=11,
    )
    return fig


major = versions.str.extract(r"^(\d+)", expand=False).fillna("missing")
share = df.assign(major=major).groupby(["quarter", "major"]).size().rename("reviews").reset_index()
fig = px.area(
    share, x="quarter", y="reviews", color="major", title="Reviews per quarter by major version"
)
mark_partial_quarter(fig).show()

### 4b. Reviews without an app version

**Concept: missing data has a mechanism, and dropping it assumes the mechanism is harmless.**

Rubin's three types: 

**MCAR** (missing completely at random): the gap is like a coin flip, unrelated to anything, and dropping those rows only costs sample size. 

**MAR** (missing at random):
the gap depends on something we observe (for example, the year: older reviews lose the field more often), so comparing within the same quarter removes the bias. 

**MNAR** (missing not at random): 
the gap depends on the very thing we care about, here possibly the user's dissatisfaction (someone who uninstalled the app, or reviews from a device where it no longer runs, may leave no version), and no observed variable can fully correct it. Analogy: a satisfaction survey left at the exit of a restaurant. If the form is missing because the stack ran out (MCAR), the answers you have are fine; if it is missing on Tuesdays (MAR), you compare Tuesdays with Tuesdays; if the angriest customers left without taking one (MNAR), every average you compute is too happy.

Why it matters here:
filtering on `app_version` (for a per-version drift chart, or a per-version training split) would silently drop a group of reviews that may be more negative than the rest, and make the app look better than it is. 

The test below compares reviews with and without a version on rating, length
and time, and then within the same quarter and length band, which rules out the simplest MAR explanations. It cannot prove the uninstall story: the data has no field that says why the version is missing.

In [ ]:
missing = df.assign(version=np.where(versions.isna(), "without version", "with version"))
missing["low_rating"] = missing["score"] <= 2

ratings = pd.crosstab(missing["version"], missing["score"], normalize="index")
display(ratings.style.format("{:.1%}"))
fig = px.bar(
    ratings.stack().rename("share").reset_index(),
    x="score",
    y="share",
    color="version",
    barmode="group",
    title="Rating distribution, with and without app version",
)
fig.update_layout(yaxis_tickformat=".0%")
fig.update_xaxes(tickmode="linear", dtick=1)
fig.show()

display(
    missing.groupby("version")["chars"]
    .describe(percentiles=[0.5, 0.9, 0.99])
    .rename(columns={"50%": "p50", "90%": "p90", "99%": "p99"})
    .round(1)
)

by_quarter = (
    missing.groupby("quarter")
    .agg(
        reviews=("score", "size"),
        without_version=("version", lambda v: v.eq("without version").mean()),
    )
    .reset_index()
)
fig = px.line(
    by_quarter,
    x="quarter",
    y="without_version",
    markers=True,
    title="Share of reviews without app version, per quarter",
)
fig.update_layout(yaxis_tickformat=".0%")
mark_partial_quarter(fig).show()

# Same quarter, so a change in the review mix over time cannot explain the gap.
low_by_quarter = missing.groupby(["quarter", "version"])["low_rating"].mean().reset_index()
fig = px.line(
    low_by_quarter,
    x="quarter",
    y="low_rating",
    color="version",
    markers=True,
    title="Share of 1-2 star reviews per quarter, with and without app version",
)
fig.update_layout(yaxis_tickformat=".0%")
mark_partial_quarter(fig).show()
gap = low_by_quarter.pivot(index="quarter", columns="version", values="low_rating")
gap = gap["without version"] - gap["with version"]
print(f"quarters where reviews without version are more negative: {(gap > 0).sum()} of {len(gap)}")

# Same length band: longer reviews are more negative, and reviews without version are longer.
bands = pd.cut(missing["chars"], [0, 10, 30, 100, 250, np.inf], right=False)
display(
    missing.groupby([bands, "version"], observed=True)["low_rating"]
    .agg(["size", "mean"])
    .unstack()
    .style.format("{:.1%}", subset=["mean"])
    .format("{:,.0f}", subset=["size"])
)

#### Conclusions on the missing versions

1. Without a version, a review is more negative and longer. The next question is whether something
else explains it, so two controls follow:
- Same quarter: reviews without a version are more negative in 26 of 27 quarters. A change in the
  review mix over time does not explain the gap.
- Same length band: the gap remains. Between 10 and 29 characters, 43% of reviews without a version
  have 1-2 stars, against 23% of those with one. Being longer does not explain it either.

The two controls are what make the result reliable. A difference between groups can come from a
third variable, which statisticians call a confounder. The comparison tests the two obvious
candidates:

- Time: what if reviews without a version were more common in a period when everyone was more
  dissatisfied? Compared within the same quarter, they stay more negative in 26 of 27 quarters.
- Length: long texts tend to be complaints; maybe "without version" only means "long text".
  Compared within the same length band, the gap remains (43% against 23% low ratings, between 10
  and 29 characters).

This technique is called **stratification**: comparing only within groups that are equal on the
suspect variable. The analogy is comparing the grades of two students from different schools:
before concluding who studies more, you compare students in the same grade. If the difference
survives, the grade was not the explanation.

2. The missing version is not random. Among reviews without a version, 50.6% have 1 star, against
25.7% among those with one. And the texts are three times longer (median of 33 characters against
11). This rules out MCAR: if it were a technical glitch at random, the two groups would look alike.

3. The missing version is linked to dissatisfaction, and it is not only an effect of time or text
length. The concept cell states what the data does not prove: no field says why the version is
missing. The hypothesis of users who uninstalled the app fits the numbers, but it is not confirmed.

## 5. Data quality problems

**Concept: data quality is measured, not assumed, and heuristics give candidates, not truth.**
Each check below is a rule of thumb with unknown precision: a "spam" flag may catch an angry but
genuine review. The point is to know the size of each problem before deciding what to do with it.
Analogy: sorting fruit at a market stall: you count the bruised ones before deciding whether to
throw them away, discount them, or make jam. Why it matters: empty and emoji-only texts are what the
API rejects (422) or will call out of distribution (`low_text_quality`), and other-language text is
the `language` detector of ADR-04. The share found here is the first estimate of how often those
guards will fire.

Edited reviews can only be detected partially: a reply dated before its review proves an edit after
the reply. That count is a **lower bound** on edits.

In [ ]:
URL = re.compile(r"https?://|www\.", re.IGNORECASE)
REPEATED_CHAR = re.compile(r"(.)\1{5,}")
LAUGH_ONLY = re.compile(r"^\s*(k{3,}|(ha){3,}|(rs){3,}|(he){3,})\s*$", re.IGNORECASE)
PT_WORDS = {
    "de",
    "que",
    "nao",
    "o",
    "a",
    "e",
    "para",
    "com",
    "um",
    "uma",
    "os",
    "no",
    "na",
    "do",
    "da",
    "eu",
    "se",
    "mais",
    "por",
    "mas",
    "muito",
    "esta",
    "app",
    "aplicativo",
    "meu",
    "ja",
    "quando",
    "consigo",
    "conta",
    "pra",
    "tem",
    "foi",
    "ser",
}
EN_WORDS = {
    "the",
    "and",
    "is",
    "it",
    "this",
    "not",
    "very",
    "good",
    "i",
    "to",
    "my",
    "can't",
    "cannot",
    "please",
    "work",
    "doesn't",
    "great",
    "bad",
    "with",
    "you",
}
ES_WORDS = {
    "el",
    "es",
    "muy",
    "pero",
    "aplicación",
    "aplicacion",
    "puedo",
    "los",
    "las",
    "bueno",
    "funciona",
    "y",
    "mi",
    "usted",
    "esta",
}


def letters_or_digits(value):
    return sum(ch.isalnum() for ch in value)


def other_language(value):
    tokens = re.findall(r"[\w']+", normalize(value))
    if len(tokens) < 3:
        return False
    pt = sum(t in PT_WORDS for t in tokens)
    en = sum(t in EN_WORDS for t in tokens)
    es = sum(t in ES_WORDS - PT_WORDS for t in tokens)
    return (en >= 2 and en > pt) or (es >= 2 and es > pt)


def canonical(value):
    # Exact-duplicate key: case, accents, punctuation and spacing ignored.
    return " ".join(re.findall(r"\w+", normalize(value)))


df["canonical"] = text.map(canonical)
checks = pd.DataFrame(
    {
        "empty_or_whitespace": text.str.strip().eq(""),
        "below_3_non_space_chars (API 422)": text.str.replace(r"\s", "", regex=True).str.len() < 3,
        "no_letters_or_digits (emoji/punctuation only)": text.map(letters_or_digits).eq(0),
        "other_language (heuristic)": text.map(other_language),
        "spam: url": text.str.contains(URL),
        "spam: repeated character x6": text.map(lambda v: bool(REPEATED_CHAR.search(v))),
        "spam: laughter only": text.str.match(LAUGH_ONLY),
        "duplicate text (exact)": text.duplicated(keep=False) & text.str.strip().ne(""),
        "duplicate text (canonical)": df["canonical"].duplicated(keep=False)
        & df["canonical"].ne(""),
        "edited after reply (lower bound)": df["reply_delay_hours"] < 0,
    }
)
quality = pd.DataFrame({"reviews": checks.sum(), "share": checks.mean()})
quality.loc["duplicate review id (removed at interim)"] = [
    meta["stats"]["duplicate_rows"],
    meta["stats"]["duplicate_rate"],
]
quality.style.format({"share": "{:.2%}", "reviews": "{:,.0f}"})

In [ ]:
# A few examples per heuristic, to judge its precision by eye (outputs are never committed).
for name in [
    "other_language (heuristic)",
    "spam: repeated character x6",
    "no_letters_or_digits (emoji/punctuation only)",
]:
    pool = printable(text[checks[name]])
    sample = pool.sample(min(5, len(pool)), random_state=0)
    print(f"--- {name}")
    for value in sample:
        print("  ", repr(value[:120]))

In [ ]:
top_dupes = (
    df[df["canonical"].ne("")]
    .groupby("canonical")
    .size()
    .sort_values(ascending=False)
    .head(15)
    .rename("reviews")
    .reset_index()
)
fig = px.bar(
    top_dupes,
    x="reviews",
    y="canonical",
    orientation="h",
    title="Most repeated texts (canonical form)",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

## 5b. Personal data inside the text

**Concept: removing the author column does not anonymize a dataset.** People write their own name,
CPF or phone number into the review, often to ask for help ("me chamo ..., meu CPF é ..."). These
are *quasi-identifiers* living inside the free text, and no column drop removes them. Analogy:
tearing the sender's name off an envelope does not help if the letter is signed. Why it matters:
the anonymized sample promised in the privacy rules must scrub the text itself, not only drop
columns; this cell counts the patterns and never prints them. Every printed example in this
notebook skips texts that match.

In [ ]:
personal = pd.DataFrame(
    {
        name: text.map(lambda v, p=pattern: bool(p.search(v)))
        for name, pattern in PERSONAL_DATA.items()
    }
)
personal["any"] = personal.any(axis=1)
pd.DataFrame({"reviews": personal.sum(), "share": personal.mean()}).style.format(
    {"reviews": "{:,.0f}", "share": "{:.2%}"}
)

## 6. Vocabulary

**Concept: Zipf's law and Heaps' law.** Word frequencies follow Zipf's law: a handful of words
("app", "não", "conta") make up a large share of all tokens, and most words appear once or twice.
Heaps' law describes the consequence: the vocabulary keeps growing as you read more text, roughly
as `V = k * N^beta` with `beta` below 1, so it never saturates. Analogy: a new city's streets:
the main avenues show up on day one, but you keep discovering new alleys every week. Why it matters:
a bag-of-words classifier only knows words it saw in training; with `beta` near 0.5 a fair share
of words in tomorrow's reviews will be new (typos, slang, new service names). The distinctive terms
per rating also show what each rating is "about", which section 8 checks against the text.

In [ ]:
TOKEN = re.compile(r"[a-z0-9]+")
STOPWORDS = {
    "a",
    "o",
    "e",
    "de",
    "do",
    "da",
    "dos",
    "das",
    "que",
    "em",
    "no",
    "na",
    "nos",
    "nas",
    "um",
    "uma",
    "para",
    "pra",
    "com",
    "por",
    "os",
    "as",
    "se",
    "eu",
    "me",
    "meu",
    "minha",
    "ao",
    "ja",
    "mais",
    "mas",
    "muito",
    "isso",
    "esse",
    "essa",
    "ele",
    "ela",
    "foi",
    "ser",
    "tem",
    "ter",
    "so",
    "sem",
    "como",
    "quando",
    "esta",
    "estou",
    "sao",
    "ou",
    "nem",
    "ate",
    "q",
    "vc",
    "voce",
    "aqui",
    "agora",
    "sempre",
    "nada",
    "tudo",
    "pois",
    "pelo",
    "pela",
}

df["tokens"] = text.map(lambda v: TOKEN.findall(normalize(v)))
ordered = df.sort_values("at")["tokens"]
vocabulary, seen_tokens, curve = set(), 0, []
for i, tokens in enumerate(ordered, start=1):
    vocabulary.update(tokens)
    seen_tokens += len(tokens)
    if i % 5000 == 0 or i == len(ordered):
        curve.append((seen_tokens, len(vocabulary)))
curve = pd.DataFrame(curve, columns=["tokens", "vocabulary"])
beta, log_k = np.polyfit(np.log(curve["tokens"]), np.log(curve["vocabulary"]), 1)
print(
    f"tokens: {seen_tokens:,}  vocabulary (accent-folded): {len(vocabulary):,}  "
    f"Heaps beta = {beta:.2f}"
)
fig = px.line(
    curve,
    x="tokens",
    y="vocabulary",
    log_x=True,
    log_y=True,
    title=f"Vocabulary growth in chronological order (Heaps beta = {beta:.2f})",
)
fig.show()

frequency = Counter(t for tokens in df["tokens"] for t in tokens)
singletons = sum(1 for c in frequency.values() if c == 1)
print(f"words seen only once: {singletons:,} ({singletons / len(frequency):.1%} of the vocabulary)")

In [ ]:
per_score = {
    s: Counter(
        t for tokens in df.loc[df["score"] == s, "tokens"] for t in tokens if t not in STOPWORDS
    )
    for s in SCORES
}


def facet_bars(frame, x, y, title):
    # Every label shown: one tick per category and enough height for each bar.
    rows = frame.groupby("score").size().max()
    fig = px.bar(
        frame,
        x=x,
        y=y,
        facet_col="score",
        orientation="h",
        title=title,
        height=max(420, 26 * rows + 120),
    )
    fig.update_yaxes(
        matches=None,
        showticklabels=True,
        tickmode="linear",
        dtick=1,
        categoryorder="total ascending",
    )
    fig.update_xaxes(matches=None)
    fig.for_each_annotation(lambda a: a.update(text=a.text.replace("score=", "")))
    return fig


top_terms = pd.DataFrame(
    [(s, term, count) for s in SCORES for term, count in per_score[s].most_common(12)],
    columns=["score", "term", "count"],
)
facet_bars(
    top_terms, "count", "term", "Most frequent single words by rating (stopwords removed)"
).show()

## 6b. Multiword expressions and collocations

**Concept: some meanings live in pairs of words, and PMI finds the pairs that stick together.**
"reconhecimento facial", "duas etapas" and "qr code" are one idea written as two words. Counting
words one at a time splits them, so "facial" and "reconhecimento" each look like separate topics.
Bigrams (pairs of adjacent words) keep them together, and keep negation attached: "não consigo"
and "não funciona" mean something that "consigo" and "funciona" alone do not. That is why the
bigram chart below does not drop "não", "nem", "nunca", "sem" and "nada" as stopwords.

To decide which pairs are real expressions rather than frequent neighbors, compare how often they
appear together with how often chance alone would put them together. That ratio, on a log scale, is
the *pointwise mutual information*: PMI(x, y) = log2 p(x, y) / (p(x) p(y)). Analogy: two people
seen together at every party are a couple; two people seen together only because both go to every
party are just regulars.

PMI has a known bias: it favors rare pairs. When both words are common, the ratio cannot get very
large, so "reconhecimento facial" (50,801 times) scores 6.5 while "fundo branco" (237 times)
scores 13.2. The normalized version, NPMI = PMI / -log2 p(x, y), runs from -1 to 1, where 1 means
"these words only ever appear together", and does not reward rarity. Pairs here are merged into
one token when NPMI >= 0.6 and the pair appears at least 200 times, so that rare accidents are not
merged.

In [ ]:
NEGATIONS = {"nao", "nem", "nunca", "sem", "nada"}
BIGRAM_STOPWORDS = STOPWORDS - NEGATIONS


def bigrams_of(tokens):
    return [
        f"{a} {b}"
        for a, b in pairwise(tokens)
        if a not in BIGRAM_STOPWORDS and b not in BIGRAM_STOPWORDS and a != b
    ]


per_score_bigrams = {
    s: Counter(b for tokens in df.loc[df["score"] == s, "tokens"] for b in bigrams_of(tokens))
    for s in SCORES
}
top_bigrams = pd.DataFrame(
    [(s, pair, count) for s in SCORES for pair, count in per_score_bigrams[s].most_common(12)],
    columns=["score", "bigram", "count"],
)
facet_bars(
    top_bigrams, "count", "bigram", "Most frequent word pairs by rating (negations kept)"
).show()

In [ ]:
MIN_PAIR_COUNT = 200
MIN_NPMI = 0.6

unigram_counts = Counter(t for tokens in df["tokens"] for t in tokens)
pair_counts = Counter(p for tokens in df["tokens"] for p in pairwise(tokens))
n_unigrams, n_pairs = sum(unigram_counts.values()), sum(pair_counts.values())

rows = []
for (a, b), n in pair_counts.items():
    if n < MIN_PAIR_COUNT or a in STOPWORDS or b in STOPWORDS or a == b:
        continue
    p_ab = n / n_pairs
    pmi = np.log2(p_ab / ((unigram_counts[a] / n_unigrams) * (unigram_counts[b] / n_unigrams)))
    rows.append((a, b, n, pmi, pmi / -np.log2(p_ab)))
collocations = pd.DataFrame(rows, columns=["first", "second", "count", "pmi", "npmi"])
collocations["merge"] = collocations["npmi"] >= MIN_NPMI
MERGE = set(
    zip(
        collocations.loc[collocations["merge"], "first"],
        collocations.loc[collocations["merge"], "second"],
        strict=True,
    )
)
print(
    f"candidate pairs (count >= {MIN_PAIR_COUNT}, no stopwords): {len(collocations):,}; "
    f"merged (NPMI >= {MIN_NPMI}): {len(MERGE)}"
)


def join_collocations(tokens):
    joined, i = [], 0
    while i < len(tokens):
        if i + 1 < len(tokens) and (tokens[i], tokens[i + 1]) in MERGE:
            joined.append(f"{tokens[i]}_{tokens[i + 1]}")
            i += 2
        else:
            joined.append(tokens[i])
            i += 1
    return joined


df["mtokens"] = df["tokens"].map(join_collocations)
display(
    collocations.sort_values("npmi", ascending=False)
    .head(25)
    .style.format({"count": "{:,.0f}", "pmi": "{:.2f}", "npmi": "{:.2f}"})
)
# PMI alone would rank by rarity: the same pairs ordered by PMI, for comparison.
collocations.sort_values("pmi", ascending=False).head(10)[
    ["first", "second", "count", "pmi", "npmi"]
].round(2)

In [ ]:
per_score_merged = {
    s: Counter(
        t for tokens in df.loc[df["score"] == s, "mtokens"] for t in tokens if t not in STOPWORDS
    )
    for s in SCORES
}
top_merged = pd.DataFrame(
    [(s, term, count) for s in SCORES for term, count in per_score_merged[s].most_common(12)],
    columns=["score", "term", "count"],
)
facet_bars(
    top_merged,
    "count",
    "term",
    "Most frequent terms by rating, collocations joined (e.g. reconhecimento_facial)",
).show()

## 6c. Distinctive terms: 1 star versus 5 stars

**Concept: a raw log-ratio overstates rare words; an informative prior shrinks them back.**
The simple log-ratio log(share in 1 star / share in 5 stars) treats a word seen 120 times, all in
1 star, almost the same as a word seen 50,000 times mostly in 1 star; the rare one can even win,
because a single extra occurrence moves its ratio a lot. Its estimate is noisy, and noise at the
extremes is what a "top 15" list selects. Analogy: a new restaurant with two five-star reviews
should not outrank one with a 4.8 average over two thousand.

The weighted log-odds with an informative Dirichlet prior (Monroe, Colaresi and Quinn, 2008) adds
pseudo-counts to every word in proportion to how common it is in all reviews, which pulls rare
words toward "no difference" until there is enough evidence. It then divides the difference by its
standard error, giving a z-score: how many standard errors the difference is from zero. Ranking by
z rewards words that are both different and well measured. The prior strength (here 10,000
pseudo-tokens spread by overall frequency) is a choice, so the cell checks how much the top of the
list moves when it changes. The tokens are the same joined tokens as in 6b, so both analyses count
words the same way.

In [ ]:
ALPHA0 = 10_000


def token_counts(mask):
    return Counter(t for tokens in df.loc[mask, "mtokens"] for t in tokens if t not in STOPWORDS)


one_star, five_star = token_counts(df["score"] == 1), token_counts(df["score"] == 5)
background = token_counts(df["score"].notna())
n_one, n_five, n_background = (sum(c.values()) for c in (one_star, five_star, background))


def weighted_log_odds(alpha0):
    rows = []
    for word, count in background.items():
        prior = alpha0 * count / n_background
        y1, y5 = one_star[word], five_star[word]
        delta = np.log((y1 + prior) / (n_one + alpha0 - y1 - prior)) - np.log(
            (y5 + prior) / (n_five + alpha0 - y5 - prior)
        )
        variance = 1 / (y1 + prior) + 1 / (y5 + prior)
        rows.append((word, delta, delta / np.sqrt(variance), y1, y5))
    return pd.DataFrame(rows, columns=["term", "log_odds", "z", "1 star", "5 stars"]).set_index(
        "term"
    )


scores_z = weighted_log_odds(ALPHA0)
top_z = pd.concat([scores_z.nlargest(15, "z"), scores_z.nsmallest(15, "z")]).reset_index()
top_z["side"] = np.where(top_z["z"] > 0, "1 star", "5 stars")
fig = px.bar(
    top_z,
    x="z",
    y="term",
    color="side",
    orientation="h",
    height=860,
    title=f"Most distinctive terms, 1 star versus 5 stars (weighted log-odds z, prior {ALPHA0:,})",
    hover_data=["1 star", "5 stars", "log_odds"],
)
fig.update_yaxes(tickmode="linear", dtick=1, categoryorder="total ascending", title=None)
fig.show()

# Sensitivity to the prior strength: overlap of the top 15 on each side.
for alpha0 in (1_000, n_background):
    other = weighted_log_odds(alpha0)
    for side, pick in (("1 star", "nlargest"), ("5 stars", "nsmallest")):
        a = set(getattr(scores_z, pick)(15, "z").index)
        b = set(getattr(other, pick)(15, "z").index)
        print(f"prior {alpha0:>9,} vs {ALPHA0:,}, {side}: {len(a & b)}/15 terms in common")

# Where the top terms of the old simple log-ratio land now.
rank = scores_z["z"].rank(ascending=False)
for word in ["colocam", "obrigam", "porco"]:
    print(
        f"{word:>8}: 1 star {one_star[word]:>4}, 5 stars {five_star[word]:>3}, "
        f"z = {scores_z.loc[word, 'z']:.2f}, rank by z = {int(rank[word])}"
    )

## 6d. Keyword in context

**Concept: a statistically strong word is a hypothesis about meaning, not the meaning.** A list of
distinctive terms says "this word is unusually common in 1-star reviews"; it does not say what
people mean by it. *Keyword in context* (KWIC) lines up every occurrence with a few words on each
side so you can read the uses side by side. Analogy: finding that "bank" is frequent in a corpus
tells you nothing until you see whether it sits next to "river" or "loan". Here it checks three
words the simple log-ratio put at the top of the 1-star list: "colocam", "obrigam" and "porco".
Texts with personal-data patterns are skipped.

In [ ]:
WINDOW, EXAMPLES = 5, 10
WORD = re.compile(r"\w+")


def kwic(term, n=EXAMPLES, window=WINDOW):
    pool = printable(text[df["tokens"].map(lambda tokens: term in tokens)])
    lines = []
    for value in pool.sample(min(n * 3, len(pool)), random_state=0):
        words = WORD.findall(value)
        hits = [i for i, w in enumerate(words) if normalize(w.lower()) == term]
        if hits:
            i = hits[0]
            lines.append(
                {
                    "left": " ".join(words[max(0, i - window) : i]),
                    "term": words[i],
                    "right": " ".join(words[i + 1 : i + 1 + window]),
                }
            )
        if len(lines) == n:
            break
    return pd.DataFrame(lines)


for term in ["colocam", "obrigam", "porco"]:
    print(f"--- {term}: {int(df['tokens'].map(lambda t, w=term: w in t).sum())} reviews")
    display(kwic(term).style.set_properties(subset=["left"], **{"text-align": "right"}))

## 7. Keyword baseline matches per theme

**Concept: coverage is not accuracy.** For each review the baseline either matches one or more
themes or abstains. The share it abstains on is its *coverage gap*; it says nothing about whether
the matches are right. That needs labels, which do not exist yet. Analogy: a smoke detector that
stays silent in 60% of rooms is not proven wrong in those rooms, and one that beeps is not proven
right; you only learn its precision by checking for fire. Why it matters: this is the first honest
number about the served model on real data, and the keyword lists were written before seeing any
review, so a theme with almost no matches is a sign the list does not speak the users' language.

In [ ]:
predictions = text.map(predict_tema)
df["tema_labels"] = predictions.map(lambda p: p[0])
df["abstained"] = df["tema_labels"].str.len().eq(0)
print(f"baseline abstains (no keyword matched) on {df['abstained'].mean():.1%} of reviews")
label_counts = df["tema_labels"].explode().dropna().value_counts().rename("reviews").to_frame()
label_counts["share_of_reviews"] = label_counts["reviews"] / len(df)
fig = px.bar(
    label_counts.reset_index(),
    x="share_of_reviews",
    y="tema_labels",
    orientation="h",
    text_auto=".1%",
    title="Share of reviews each theme matches (multi-label)",
)
fig.update_layout(
    yaxis={"categoryorder": "total ascending"}, xaxis_tickformat=".0%", yaxis_title=None
)
fig.show()
by_score = df.groupby("score")["abstained"].mean().rename("abstention_rate").to_frame()
display(by_score.style.format("{:.1%}"))
label_counts

## 8. Coherence between rating and text

**Concept: the rating is a noisy proxy, not a label.** A 5-star review can describe a bug ("great
app, but it crashes on login") and a 1-star review can be praise typed in the wrong place. The
mismatch rate is a measure of *label noise* if someone were tempted to use stars as labels.
Analogy: a thermometer that is right most days but occasionally reads the sun on the window instead
of the room. Why it matters: it is the evidence behind ADR-01 (rating is never a feature) and it
tells the annotation rubric which cases need care. The baseline outcome is used as a rough reading
of the text: "problem" when any problem theme matched, "praise only" when only `nenhum` matched.

In [ ]:
def outcome(labels):
    if not labels:
        return "no match"
    return "praise only" if labels == ["nenhum"] else "problem"


df["text_reading"] = df["tema_labels"].map(outcome)
crosstab = pd.crosstab(df["score"], df["text_reading"], normalize="index")
fig = px.imshow(
    crosstab,
    text_auto=".0%",
    aspect="auto",
    color_continuous_scale="Blues",
    title="Text reading (baseline) by rating, share of each rating row",
)
fig.show()
incoherent = {
    "5 stars, text matches a problem": ((df["score"] == 5) & (df["text_reading"] == "problem")),
    "1 star, text is praise only": ((df["score"] == 1) & (df["text_reading"] == "praise only")),
}
for name, mask in incoherent.items():
    print(f"{name}: {mask.sum():,} ({mask.mean():.2%} of all reviews)")
    pool = printable(text[mask])
    for value in pool.sample(min(3, len(pool)), random_state=1):
        print("    ", repr(value[:140]))

## 9. Effect of each cleaning step

**Concept: every transformation is either reversible or lossy, and you should know which before
applying it.** Trimming spaces loses nothing; lowercasing loses a little (ALL CAPS anger); removing
accents merges different words in Portuguese ("é" and "e", "está" and "esta"). A good cleaning step
is also *idempotent*: applying it twice gives the same result as once. Analogy: washing vegetables
removes dirt, peeling also removes vitamins: check what each step throws away. Why it matters: the
interim table keeps the text exactly as written; this section measures, step by step, how many
texts each candidate step changes and whether accents survive, so that the cleaning decision is
made on evidence. It also checks for *mojibake* (text decoded with the wrong encoding, like "Ã§"
instead of "ç"), which would mean accents were already broken at collection.

In [ ]:
ACCENTED = re.compile(r"[áàâãéêíóôõúüçÁÀÂÃÉÊÍÓÔÕÚÜÇ]")
MOJIBAKE = re.compile(r"Ã[\x80-\xbf§£©ª¡³µ]|Â\s")
EMOJI_OR_SYMBOL = lambda v: "".join(  # noqa: E731
    ch
    for ch in v
    if not unicodedata.category(ch).startswith(("So", "Sk", "Cf", "Cs"))
    and ch not in "\ufe0f\u200d"
)

steps = [
    ("strip", str.strip),
    ("collapse whitespace", lambda v: re.sub(r"\s+", " ", v)),
    ("unicode NFC", lambda v: unicodedata.normalize("NFC", v)),
    ("lowercase", str.lower),
    ("remove urls", lambda v: re.sub(r"https?://\S+|www\.\S+", " ", v)),
    ("remove emoji and symbols", EMOJI_OR_SYMBOL),
    ("baseline normalize (strip accents)", normalize),
]

current, rows, examples = text.copy(), [], {}
print(f"mojibake candidates in the collected text: {text.str.contains(MOJIBAKE).sum():,}")
for name, step in steps:
    after = current.map(step)
    changed = after.ne(current)
    rows.append(
        {
            "step": name,
            "texts_changed": int(changed.sum()),
            "share_changed": changed.mean(),
            "texts_with_accents_after": after.str.contains(ACCENTED).mean(),
            "mean_chars_after": after.str.len().mean(),
            "idempotent": bool(after.map(step).eq(after).all()),
        }
    )
    shown = changed & ~text.map(has_personal_data)
    examples[name] = list(zip(current[shown].head(2), after[shown].head(2), strict=False))
    current = after
effect = pd.DataFrame(rows).set_index("step")
display(
    effect.style.format(
        {
            "share_changed": "{:.1%}",
            "texts_with_accents_after": "{:.1%}",
            "mean_chars_after": "{:.1f}",
        }
    )
)
for name, pairs in examples.items():
    print(f"--- {name}")
    for before, after in pairs:
        print(f"   before: {before[:90]!r}\n   after:  {after[:90]!r}")

In [ ]:
merged = (
    pd.DataFrame({"original": text.str.lower(), "folded": text.map(normalize)})
    .assign(tokens=lambda d: d["original"].str.findall(r"\w+"))
    .explode("tokens")["tokens"]
    .dropna()
)
collisions = (
    pd.DataFrame({"word": merged, "folded": merged.map(normalize)})
    .drop_duplicates()
    .groupby("folded")["word"]
    .agg(list)
)
collisions = collisions[collisions.str.len() > 1]
print(f"distinct folded forms that merge two or more spellings: {len(collisions):,}")
examples = ["e", "esta", "nao", "sao", "ja", "voce", "so", "codigo", "numero"]
collisions.reindex([w for w in examples if w in collisions.index]).to_frame("spellings merged")

## 10. Leakage risks for the future split

**Concept: leakage is when the test set contains information the model already saw in training,
so the score measures memory instead of generalization.** Analogy: a student who finds next week's
exam questions in the study guide scores well and has learned nothing. Two routes matter here:

- **Repeated and near-repeated texts.** "Ótimo" appears thousands of times; a random split puts
  copies on both sides, and the model gets them right by memory. Near-duplicates ("não consigo
  entrar" versus "nao consigo entrar!!") do the same. The fix is to split by *groups* of identical
  or near-identical text, so all copies land on the same side.
- **Time.** A random split trains on 2025 reviews and tests on 2022 ones, so the model has seen
  the future (new services, new error messages) when predicting the past. In production it will
  always predict the future from the past. A temporal holdout (test on the most recent months)
  measures the situation the API is actually in.

Calibration (40%) and test (60%) of ADR-07 come from the hand-labeled set, but the same rules apply
to how that set is drawn and how it relates to the LLM-labeled training set.

In [ ]:
rng = np.random.default_rng(0)
is_test = rng.random(len(df)) < 0.2
train_keys = set(df.loc[~is_test, "canonical"])
in_train = df.loc[is_test, "canonical"].isin(train_keys)
long_enough = df.loc[is_test, "words"] >= 5
print(f"random 80/20 split: {in_train.mean():.1%} of test reviews have a canonical twin in train")
print(f"  among test reviews with 5+ words: {in_train[long_enough].mean():.1%}")

sample = df.loc[df["words"] >= 5].sample(min(30000, int((df["words"] >= 5).sum())), random_state=0)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

vectors = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2).fit_transform(
    sample["text"].map(normalize)
)
distances, neighbors = (
    NearestNeighbors(n_neighbors=2, metric="cosine").fit(vectors).kneighbors(vectors)
)
similarity = 1 - distances[:, 1]
neighbor_canonical = sample["canonical"].to_numpy()[neighbors[:, 1]]
near_not_exact = (similarity >= 0.9) & (neighbor_canonical != sample["canonical"].to_numpy())
print(
    f"in a sample of {len(sample):,} reviews with 5+ words: "
    f"{(similarity >= 0.9).mean():.1%} have a neighbor with cosine >= 0.9, "
    f"and {near_not_exact.mean():.1%} of the sample are near-duplicates the canonical form misses"
)
safe = ~sample["text"].map(has_personal_data).to_numpy()
for i in [j for j in np.flatnonzero(near_not_exact) if safe[j] and safe[neighbors[j, 1]]][:4]:
    print(f"   {sample['text'].iloc[i][:80]!r}\n   {sample['text'].iloc[neighbors[i, 1]][:80]!r}\n")

In [ ]:
yearly = (
    df.assign(year=df["at"].dt.year)
    .groupby("year")
    .agg(
        reviews=("review_id", "size"),
        mean_score=("score", "mean"),
        median_chars=("chars", "median"),
        baseline_abstention=("abstained", "mean"),
    )
)
themes_by_year = (
    df.assign(year=df["at"].dt.year)
    .explode("tema_labels")
    .dropna(subset=["tema_labels"])
    .groupby(["year", "tema_labels"])
    .size()
    .div(df.assign(year=df["at"].dt.year).groupby("year").size(), level="year")
    .rename("share")
    .reset_index()
)
fig = px.line(
    themes_by_year,
    x="year",
    y="share",
    color="tema_labels",
    markers=True,
    title="Share of reviews matching each theme, by year (shift over time)",
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()
yearly.style.format(
    {"mean_score": "{:.2f}", "median_chars": "{:.0f}", "baseline_abstention": "{:.1%}"}
)

## 10b. How much is left after deduplication

**Concept: deduplicating changes the class mix, and precision depends on the class mix.** Short
praise is copied thousands of times ("ótimo", "muito bom"), while complaints are mostly written
once. Counting each distinct text once therefore shrinks praise and inflates complaints. That is
what we want for annotation, where a thousand copies of "ótimo" teach nothing, but it is not the
world the API lives in, where every copy arrives as a request.

The catch is *prevalence*. Precision is TP / (TP + FP), and the number of false positives grows
with the number of negatives the model sees. A threshold tuned to precision 0.80 on a set where
complaints are 70% of the items can fall well below 0.80 in production, where they are 40%,
because there are many more negatives to be wrong about. Analogy: a medical test with the same
sensitivity and specificity has a far lower positive predictive value in a population where the
disease is rare than in a hospital ward where it is common.

So the decision for ADR-07: annotate unique texts sampled to favor complaints, record each item's
selection probability, and compute calibration and test metrics with weights (inverse selection
probability times how many reviews share the text). The thresholds are then chosen for the traffic
the API actually serves, not for the sample we found convenient to label.

In [ ]:
labeled = df[df["canonical"].ne("")]
groups = labeled.groupby("canonical").agg(
    copies=("review_id", "size"),
    score=("score", lambda s: int(s.mode().iloc[0])),
    chars=("chars", "first"),
    words=("words", "first"),
    text_reading=("text_reading", "first"),
)


def unique_summary(mask_reviews):
    return {
        "reviews": int(mask_reviews.sum()),
        "unique texts": int(labeled.loc[mask_reviews, "canonical"].nunique()),
        "unique texts, 5+ words": int(
            labeled.loc[mask_reviews & (labeled["words"] >= 5), "canonical"].nunique()
        ),
    }


unique_rows = {"all": unique_summary(labeled["score"].notna())}
unique_rows["1 and 2 stars"] = unique_summary(labeled["score"].isin([1, 2]))
for s in SCORES:
    unique_rows[f"{s} stars"] = unique_summary(labeled["score"].eq(s))
uniques = pd.DataFrame(unique_rows).T
uniques["unique share"] = uniques["unique texts"] / uniques["reviews"]
uniques.style.format(
    {
        "reviews": "{:,.0f}",
        "unique texts": "{:,.0f}",
        "unique texts, 5+ words": "{:,.0f}",
        "unique share": "{:.1%}",
    }
)

In [ ]:
# Length of each distinct text (one row per text) against every review (one row per copy).
QS = [0.10, 0.50, 0.90, 0.99]
length_table = pd.DataFrame(
    {
        "all reviews": labeled["chars"].quantile(QS).to_numpy(),
        "unique texts": groups["chars"].quantile(QS).to_numpy(),
        "unique texts, 1-2 stars": groups.loc[groups["score"].isin([1, 2]), "chars"]
        .quantile(QS)
        .to_numpy(),
    },
    index=[f"p{int(q * 100)}" for q in QS],
)
display(length_table.round(0))

bins = np.arange(0, 510, 10)


def binned(values, name):
    shares = pd.cut(values.clip(upper=499), bins=bins, right=False).value_counts(
        normalize=True, sort=False
    )
    return pd.DataFrame(
        {"chars": [b.left for b in shares.index], "share": shares.to_numpy(), "set": name}
    )


compare = pd.concat(
    [binned(labeled["chars"], "all reviews"), binned(groups["chars"], "unique texts")]
)
fig = px.line(
    compare,
    x="chars",
    y="share",
    color="set",
    title="Length distribution: every review versus each distinct text once",
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

In [ ]:
# How deduplication moves the class mix: rating and baseline reading, per review versus per text.
mix = pd.concat(
    [
        labeled["score"].value_counts(normalize=True).rename("per review"),
        groups["score"].value_counts(normalize=True).rename("per unique text"),
    ],
    axis=1,
).reindex(SCORES)
reading = pd.concat(
    [
        labeled["text_reading"].value_counts(normalize=True).rename("per review"),
        groups["text_reading"].value_counts(normalize=True).rename("per unique text"),
    ],
    axis=1,
)
display(mix.style.format("{:.1%}"))
display(reading.style.format("{:.1%}"))
fig = px.bar(
    mix.reset_index().melt(id_vars="score", var_name="counted", value_name="share"),
    x="score",
    y="share",
    color="counted",
    barmode="group",
    title="Rating mix counted per review versus per unique text",
)
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

## 11. Numbers to carry forward

**Concept: an exploration ends with decisions' inputs, not with charts.** Each number below feeds a
later step: the contract (p99), the guards (quality shares), the evaluation design (leakage), and
the honest README section on what the baseline does.

In [ ]:
pd.Series(
    {
        "complete history": meta["complete_history"],
        "reviews (interim)": f"{len(df):,}",
        "oldest review (UTC, last edit)": df["at"].min().isoformat(),
        "newest review (UTC, last edit)": df["at"].max().isoformat(),
        "duplicate rate by review id": f"{meta['stats']['duplicate_rate']:.4%}",
        "p99 length, characters": f"{P99_CHARS:.0f}",
        "share at the 500-character cap": f"{(df['chars'] >= 500).mean():.2%}",
        "share with no letters or digits": f"{checks.iloc[:, 2].mean():.2%}",
        "share other language (heuristic)": f"{checks.iloc[:, 3].mean():.2%}",
        "share with a canonical duplicate": f"{checks['duplicate text (canonical)'].mean():.1%}",
        "share with a personal-data pattern in the text": f"{personal['any'].mean():.2%}",
        "unique texts (normalized)": f"{uniques.loc['all', 'unique texts']:,}",
        "unique texts, 1 and 2 stars": f"{uniques.loc['1 and 2 stars', 'unique texts']:,}",
        "1-2 star share per review": f"{mix.loc[[1, 2], 'per review'].sum():.1%}",
        "1-2 star share per unique text": f"{mix.loc[[1, 2], 'per unique text'].sum():.1%}",
        "baseline abstention rate": f"{df['abstained'].mean():.1%}",
        "random split, test with twin in train": f"{in_train.mean():.1%}",
        "collocations joined (NPMI >= 0.6, 200+ occurrences)": f"{len(MERGE)}",
        "Heaps beta": f"{beta:.2f}",
    }
).to_frame("value")